# Logger tests

This notebook exercises the queue-backed logger and its sensitive-data redaction behavior.

In [1]:
import sys
import tempfile
import time
import uuid
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "boti").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

In [2]:
from boti.core import Logger
from boti.core.models import LoggerConfig

with tempfile.TemporaryDirectory() as tmp_dir:
    suffix = uuid.uuid4().hex[:8]
    config = LoggerConfig(
        log_dir=Path(tmp_dir),
        logger_name=f"notebook_logger_{suffix}",
        log_file=f"notebook_logger_{suffix}",
    )
    logger = Logger(config)
    logger.set_level(Logger.INFO)
    logger.info("User password is hunter2")
    logger.info("Service token is abc123")
    logger.info("regular message")

    time.sleep(0.5)

    log_path = Path(tmp_dir) / f"notebook_logger_{suffix}.log"
    content = log_path.read_text(encoding="utf-8")
    assert "[REDACTED SENSITIVE DATA]" in content
    assert "hunter2" not in content
    assert "abc123" not in content
    assert "regular message" in content

print(content)

[2026-06-22 10:31:36][INFO][notebook_logger_cd588a3f] [REDACTED SENSITIVE DATA]
[2026-06-22 10:31:36][INFO][notebook_logger_cd588a3f] [REDACTED SENSITIVE DATA]
[2026-06-22 10:31:36][INFO][notebook_logger_cd588a3f] regular message
[2026-06-22 10:31:36][INFO][notebook_logger_cd588a3f] [REDACTED SENSITIVE DATA]
[2026-06-22 10:31:36][INFO][notebook_logger_cd588a3f] [REDACTED SENSITIVE DATA]
[2026-06-22 10:31:36][INFO][notebook_logger_cd588a3f] regular message

